# Mixtral-8x7B MoE-Only Cross-Arch — Isolating the MoE Contribution

**Goal**: isolate **which** architectural component enables amnesic in Qwen3.6-35B-A3B.

Current evidence:
- Qwen3.6-35B-A3B (MoE + GDN + Gated-Attn): **+2.7 pp Pareto** ✓
- Qwen2.5-32B (dense, standard attn): **−4.5 pp** ✗
- 7.2 pp architecture gap

**Missing control**: is it **MoE routing alone** that enables the effect, or does it require GDN state + Gated-Attention specifically?

**Test**: Mixtral-8x7B-Instruct-v0.1 — **MoE-only** (8 experts, 2 active) with **standard GQA attention** and **standard MLP**. No GDN. No Gated-Attention.

**Decision tree**:
- **Mixtral +2pp+ Pareto** → **MoE routing is sufficient** for amnesic. Paper claim: "MoE-enabled observational channels."
- **Mixtral null (0 pp)** → MoE alone is insufficient; GDN state or Gated-Attention is critical.
- **Mixtral negative (−2pp+)** → amnesic requires the Qwen3.6-specific combination.

Each outcome narrows the mechanistic claim.

**Model**: `mistralai/Mixtral-8x7B-Instruct-v0.1`
- 32 layers, d_model 4096
- MoE: 8 experts, 2 active per token (vs Qwen3.6's 128/8)
- Standard GQA attention (no Gated-Attention)
- Standard MLP (no GDN)
- ~90 GB bf16, 47B total / 13B active params

**Budget**: ~2-3h on RTX 6000 Blackwell 96 GB.

## Cell 1 — Install + setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
except Exception:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', 'transformers', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping: {e})')

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/mixtral_stage_b'); OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load Mixtral-8x7B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'mistralai/Mixtral-8x7B-Instruct-v0.1'
# 32 layers, 8 experts, 2 active per token

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','layers'), ('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

def get_all_layers(m):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','layers'), ('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                return cur
    raise RuntimeError('layers not found')

all_layers = get_all_layers(model)
N_LAYERS = len(all_layers)
PROBE_LAYERS = [N_LAYERS // 3, N_LAYERS // 2, (2 * N_LAYERS) // 3]
print(f'{MODEL_ID}')
print(f'  Layers: {N_LAYERS}')
print(f'  Probe layers (33%/50%/67%): {PROBE_LAYERS}')
print(f'  VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Load SuperGPQA 10-opt questions

In [ ]:
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        opts = json.loads(meta['options'])
        if len(opts) == 10:
            questions.append(dict(
                question=meta['question'], options=opts,
                gold_letter=meta['gold_letter'], n_options=10))
print(f'{len(questions)} 10-opt questions')

def format_mcq(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = (
        "Answer the following multiple-choice question. Reason briefly, "
        "then give the letter of the correct answer in \\boxed{}.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
def extract_letter(c, n_opts):
    m = list(BOXED_RE.finditer(c))
    if m:
        l = m[-1].group(1).upper()
        if ord(l)-ord('A') < n_opts: return l
    tail = c[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands and ord(cands[-1])-ord('A') < n_opts: return cands[-1]
    return None

letter_ids = {L: tok(L, add_special_tokens=False).input_ids[0] for L in 'ABCDEFGHIJ'}

## Cell 4 — Generate rollouts + capture residuals (~2-3h)

In [ ]:
captured_full = {L: None for L in PROBE_LAYERS}
def make_capture_full(L):
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        captured_full[L] = h.detach().clone()
    return hook

cap_handles = [get_layer_module(model, L).register_forward_hook(make_capture_full(L)) for L in PROBE_LAYERS]

N_ROLLOUTS = 3
MAX_NEW = 200
TEMP = 0.7
POOL_WINDOW = 10

rollouts_data = []
random.seed(42)
torch.manual_seed(42)

t0 = time.time()
for q_idx, q in enumerate(tqdm(questions, desc='rollouts')):
    p = format_mcq(q['question'], q['options'])
    prompt_ids = tok(p, return_tensors='pt').input_ids.to('cuda')
    if prompt_ids.shape[1] > 3800: continue
    prompt_len = prompt_ids.shape[1]
    for r in range(N_ROLLOUTS):
        try:
            torch.manual_seed(42 + q_idx * 7 + r)
            with torch.no_grad():
                gen_out = model.generate(
                    input_ids=prompt_ids,
                    attention_mask=torch.ones_like(prompt_ids),
                    max_new_tokens=MAX_NEW,
                    do_sample=True, temperature=TEMP,
                    pad_token_id=tok.pad_token_id, use_cache=True)
            full_ids = gen_out[0]
            response_ids = full_ids[prompt_len:].tolist()
            response_text = tok.decode(response_ids, skip_special_tokens=True)
            pred_letter = extract_letter(response_text, q['n_options'])
            if pred_letter is None: continue
            seq = torch.tensor([prompt_ids[0].tolist() + response_ids[:POOL_WINDOW]], device='cuda')
            if seq.shape[1] > 4096: continue
            with torch.no_grad():
                _ = model(input_ids=seq, attention_mask=torch.ones_like(seq))
            acts_pooled = {}
            for L in PROBE_LAYERS:
                response_acts = captured_full[L][0, prompt_len:prompt_len+POOL_WINDOW, :]
                if response_acts.shape[0] < POOL_WINDOW:
                    acts_pooled[L] = None; break
                acts_pooled[L] = response_acts.mean(dim=0).float().cpu().numpy()
            if any(v is None for v in acts_pooled.values()): continue
            rollouts_data.append(dict(
                q_idx=q_idx, rollout_id=r,
                gold=q['gold_letter'], pred=pred_letter,
                correct=(pred_letter == q['gold_letter']),
                n_options=q['n_options'],
                question=q['question'], options=q['options'],
                **{f'act_L{L}': v for L, v in acts_pooled.items()}))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); continue
        except Exception as e:
            continue
    if (q_idx + 1) % 20 == 0:
        total = len(rollouts_data)
        acc = sum(1 for r in rollouts_data if r['correct']) / max(total, 1)
        print(f'  q{q_idx+1}: {total} rollouts, acc {acc*100:.1f}%, {(time.time()-t0)/60:.1f} min')
        with open(OUT/'rollouts_partial.pkl', 'wb') as f:
            pickle.dump(rollouts_data, f)

for h in cap_handles: h.remove()

print(f'\n{len(rollouts_data)} total rollouts in {(time.time()-t0)/60:.1f} min')
if rollouts_data:
    acc = sum(1 for r in rollouts_data if r['correct']) / len(rollouts_data) * 100
    print(f'Rollout accuracy: {acc:.1f}%')
    letter_dist = {L: sum(1 for r in rollouts_data if r['pred']==L) for L in 'ABCDEFGHIJ'}
    print(f'Letter distribution: {letter_dist}')

with open(OUT/'rollouts.pkl', 'wb') as f:
    pickle.dump(rollouts_data, f)

## Cell 5 — Fit probes at each layer

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

q_idxs = sorted(set(r['q_idx'] for r in rollouts_data))
random.Random(42).shuffle(q_idxs)
cut = int(0.7 * len(q_idxs))
train_q = set(q_idxs[:cut])
test_q = set(q_idxs[cut:])
train_rollouts = [r for r in rollouts_data if r['q_idx'] in train_q]
test_rollouts = [r for r in rollouts_data if r['q_idx'] in test_q]
print(f'Train: {len(train_rollouts)} ({len(train_q)} Q) | Test: {len(test_rollouts)} ({len(test_q)} Q)')

letters = sorted(set(r['pred'] for r in train_rollouts))
letter_to_int = {l: i for i, l in enumerate(letters)}
print(f'Letters: {letters}')

d_stacks = {}
probe_info = {}
for L in PROBE_LAYERS:
    X_tr = np.stack([r[f'act_L{L}'] for r in train_rollouts])
    y_tr = np.array([letter_to_int[r['pred']] for r in train_rollouts])
    scaler = StandardScaler().fit(X_tr)
    pca = PCA(n_components=min(128, X_tr.shape[0]-1), random_state=42).fit(scaler.transform(X_tr))
    lr = LogisticRegression(C=0.5, max_iter=3000, random_state=42).fit(
        pca.transform(scaler.transform(X_tr)), y_tr)
    train_score = lr.score(pca.transform(scaler.transform(X_tr)), y_tr)
    X_te = np.stack([r[f'act_L{L}'] for r in test_rollouts])
    y_te = np.array([letter_to_int[r['pred']] if r['pred'] in letter_to_int else -1 for r in test_rollouts])
    mask = y_te >= 0
    test_score = lr.score(pca.transform(scaler.transform(X_te[mask])), y_te[mask]) if mask.sum() > 0 else 0.0
    probe_info[L] = dict(train=train_score, test=test_score)
    print(f'L{L}: train {train_score:.3f} | test {test_score:.3f}')

    dirs = []
    for li in range(len(letters)):
        coef = lr.coef_[li]
        d_scaled = pca.components_.T @ coef
        d_raw = d_scaled * scaler.scale_
        d_raw = d_raw / (np.linalg.norm(d_raw) + 1e-8)
        dirs.append(torch.from_numpy(d_raw.astype(np.float32)).cuda().to(torch.bfloat16))
    d_stacks[L] = torch.stack(dirs)

print(f'\n✅ probes + directions for {PROBE_LAYERS}')

## Cell 6 — Install amnesic hooks + eval

In [ ]:
class AmnesicHook:
    def __init__(self, layer_idx):
        self.layer_idx = layer_idx
        self.mode = 'off'
        self.prompt_len = None
    def set(self, prompt_len):
        self.mode = 'on'; self.prompt_len = prompt_len
    def off(self):
        self.mode = 'off'
    def make_hook(self):
        def hook(module, inp, out):
            if self.mode == 'off': return out
            hidden = out[0] if isinstance(out, tuple) else out
            T = hidden.shape[1]
            if T != self.prompt_len: return out
            hidden = hidden.clone()
            x = hidden[:, -1, :]
            ds = d_stacks[self.layer_idx]
            coefs = x @ ds.T
            delta = coefs @ ds
            hidden[:, -1, :] = x - delta
            if isinstance(out, tuple): return (hidden,) + out[1:]
            return hidden
        return hook

hooks = {L: AmnesicHook(L) for L in PROBE_LAYERS}
handles = {L: get_layer_module(model, L).register_forward_hook(hooks[L].make_hook()) for L in PROBE_LAYERS}
print(f'amnesic hooks on {PROBE_LAYERS}')

seen = set(); test_questions = []
for r in rollouts_data:
    if r['q_idx'] in test_q and r['q_idx'] not in seen:
        seen.add(r['q_idx'])
        test_questions.append(r)
print(f'{len(test_questions)} test questions')

def format_mcq_direct(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = ("Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

eval_results = []
for r in tqdm(test_questions, desc='eval'):
    try:
        p = format_mcq_direct(r['question'], r['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        prompt_len = ids.shape[1]
        row = dict(q_idx=r['q_idx'], gold=r['gold'], n_options=r['n_options'])
        for L in PROBE_LAYERS: hooks[L].off()
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
        logits = out.logits[0, -1]
        letter_logits = {LL: logits[letter_ids[LL]].item() for LL in 'ABCDEFGHIJ'[:r['n_options']]}
        row['baseline_pred'] = max(letter_logits, key=letter_logits.get)
        row['baseline_correct'] = (row['baseline_pred'] == r['gold'])
        for L in PROBE_LAYERS:
            for Lo in PROBE_LAYERS: hooks[Lo].off()
            hooks[L].set(prompt_len)
            with torch.no_grad():
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]
            letter_logits = {LL: logits[letter_ids[LL]].item() for LL in 'ABCDEFGHIJ'[:r['n_options']]}
            pred = max(letter_logits, key=letter_logits.get)
            row[f'amn_L{L}_pred'] = pred
            row[f'amn_L{L}_correct'] = (pred == r['gold'])
        for L in PROBE_LAYERS: hooks[L].off()
        eval_results.append(row)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

print(f'\n{len(eval_results)} test evals complete')

## Cell 7 — Analysis + architectural verdict

In [ ]:
from scipy.stats import binomtest

print(f'=== {MODEL_ID} (MoE-only, no GDN, no Gated-Attn) ===')
print(f'N_test = {len(eval_results)} questions\n')

bc = [r['baseline_correct'] for r in eval_results]
base_acc = sum(bc) / len(eval_results) * 100
print(f'Mixtral baseline: {base_acc:.1f}%')
print(f'Qwen3.6 hybrid reference baseline: 53.6%')
print(f'Qwen2.5-32B dense reference baseline: 40.9%\n')

print(f'{"layer":>6s}  {"acc%":>6s}  {"Δpp":>6s}  {"gain":>5s}  {"lost":>5s}  {"p":>7s}  {"probe_test":>10s}')
stats = []
for L in PROBE_LAYERS:
    cc = [r[f'amn_L{L}_correct'] for r in eval_results]
    acc = sum(cc) / len(eval_results)
    delta = (acc - base_acc/100) * 100
    g = sum(1 for b, c in zip(bc, cc) if not b and c)
    l = sum(1 for b, c in zip(bc, cc) if b and not c)
    p = binomtest(g, g+l, p=0.5, alternative='two-sided').pvalue if g+l > 0 else 1.0
    stats.append((L, acc*100, delta, g, l, p))
    star = ' ***' if delta>2.5 and l==0 else (' **' if delta>1.5 and l==0 else (' *' if delta>0.5 and l==0 else ''))
    print(f'{"L"+str(L):>6s}  {acc*100:5.1f}%  {delta:+5.1f}  {g:>5d}  {l:>5d}  {p:>7.3f}  {probe_info[L]["test"]:>10.3f}{star}')

best_pareto = max([s for s in stats if s[4] == 0], key=lambda s: s[2], default=None)
best_overall = max(stats, key=lambda s: s[2])

# Architecture comparison table
print('\n=== Architecture comparison (cross-model) ===')
print(f'{"model":30s}  {"arch":20s}  {"baseline":>8s}  {"best_Pareto":>11s}')
print(f'{"Qwen3.6-35B-A3B":30s}  {"MoE+GDN+GatedAttn":20s}  {"53.6%":>8s}  {"+2.7pp":>11s}')
print(f'{"Qwen2.5-32B-Instruct":30s}  {"dense":20s}  {"40.9%":>8s}  {"-4.5pp":>11s}')
mx_pareto = f'{best_pareto[2]:+.1f}pp' if best_pareto else 'none'
mx_best = f'{best_overall[2]:+.1f}pp'
print(f'{"Mixtral-8x7B-Instruct":30s}  {"MoE-only (no GDN)":20s}  {base_acc:>7.1f}%  {mx_pareto:>11s}')

print('\n=== VERDICT — which architectural component enables amnesic? ===')
if best_pareto and best_pareto[2] >= 2.0:
    print(f'*** MoE ROUTING IS SUFFICIENT: Mixtral +{best_pareto[2]:.1f}pp Pareto')
    print('    GDN and Gated-Attention are NOT required.')
    print('    Paper claim: "MoE routing creates observational channels separable from causal signal"')
elif best_pareto and best_pareto[2] > 0.5:
    print(f'**  MoE WEAK ENABLER: Mixtral {best_pareto[2]:+.1f}pp Pareto (vs +2.7pp hybrid)')
    print('    MoE alone gives a smaller effect; GDN or Gated-Attn amplifies.')
    print('    Paper claim: "MoE is necessary but not sufficient for full amnesic"')
elif best_overall[2] > -1.5:
    print(f'-- MoE ALONE INSUFFICIENT: Mixtral best {best_overall[2]:+.1f}pp (non-Pareto, {best_overall[4]} lost)')
    print('    MoE routing alone does NOT replicate amnesic.')
    print('    Paper claim: "GDN state or Gated-Attention is the key enabler"')
else:
    print(f'XX MoE ACTIVELY HURTS like dense: {best_overall[2]:+.1f}pp')
    print('    Mixtral behaves like dense; Qwen3.6 uniqueness is GDN + Gated-Attn combo.')
    print('    Paper claim: "Amnesic requires GDN state or Gated-Attention specifically"')

print(f'\nBaseline sanity: {base_acc:.1f}% (target ≥30% for valid test)')
if base_acc < 25:
    print('WARNING: baseline may be too low for clean test')

# Print probe quality summary
print(f'\nProbe quality:')
for L in PROBE_LAYERS:
    print(f'  L{L}: train {probe_info[L]["train"]:.3f} | test {probe_info[L]["test"]:.3f}')